# 🧬 GFP生成 - 交互式Notebook

这个notebook提供了一个交互式界面来生成和分析GFP候选序列。

**功能：**
- 交互式生成参数调整
- 实时生成进度显示
- 可视化分析结果
- 候选序列比对和分析

## 📦 1. 导入依赖和设置环境

In [ ]:
import sys
import os

# 添加项目路径
project_root = os.path.abspath('..')
sys.path.insert(0, project_root)

# 添加ESM路径
esm_path = '/mnt/disk3/tio_nekton4/esm3/esm'
sys.path.insert(0, esm_path)

print(f"项目根目录: {project_root}")
print(f"ESM路径: {esm_path}")

In [ ]:
# 导入标准库
import numpy as np
import pandas as pd
import pickle
from collections import Counter
import time

# 导入绘图库
import matplotlib.pyplot as plt
import seaborn as sns

# 设置绘图风格
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ 标准库导入完成")

In [ ]:
# 导入项目模块
from config import *
from utils.esm_wrapper import ESM3Generator
from utils.structure_utils import (
    load_pdb, 
    sequence_identity, 
    save_to_fasta,
    load_from_fasta
)
from utils.evaluation import (
    evaluate_candidate,
    rank_candidates,
    generate_summary_stats
)

print("✓ 项目模块导入完成")

In [ ]:
# 检查GPU
import torch

print(f"CUDA可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU名称: {torch.cuda.get_device_name(0)}")
    print(f"显存总量: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## 🔧 2. 初始化和配置

In [ ]:
# 初始化ESM3生成器（这一步会比较慢）
print("初始化ESM3生成器...")
print("提示: 第一次加载可能需要3-5分钟，请耐心等待")

generator = ESM3Generator(
    model_dir=MODEL_DIR,
    model_name=MODEL_NAME
)

print("✓ 生成器初始化完成")

In [ ]:
# 加载或创建prompt
prompt_file = os.path.join(PROMPTS_DIR, 'gfp_prompt.pkl')

if os.path.exists(prompt_file):
    print("加载现有prompt...")
    with open(prompt_file, 'rb') as f:
        prompt_data = pickle.load(f)
    print("✓ Prompt加载完成")
else:
    print("⚠ Prompt文件不存在")
    print("请先运行: python scripts/02_create_prompt.py")
    prompt_data = None

if prompt_data:
    print(f"\nPrompt信息:")
    print(f"  序列长度: {len(prompt_data['sequence'])}")
    print(f"  关键残基: {len(prompt_data['key_residues'])} 个")
    print(f"  Mask位置: {prompt_data['sequence'].count('_')} 个")
    print(f"  序列示例: {prompt_data['sequence'][:60]}...")

## 🎨 3. 交互式生成参数

In [ ]:
# 设置生成参数（可以修改这些值）
generation_params = {
    'temperature': 0.7,              # 温度: 0.3-1.5
    'structure_steps': 200,          # 结构生成步数
    'sequence_steps': 150,           # 序列生成步数
    'num_candidates': 5,             # 要生成的候选数量
}

print("当前生成参数:")
for key, value in generation_params.items():
    print(f"  {key}: {value}")

print("\n参数说明:")
print("  temperature: 越高越多样，越低越保守 (建议: 0.5-1.0)")
print("  structure_steps: 结构生成精度 (建议: 100-300)")
print("  sequence_steps: 序列生成精度 (建议: 75-200)")
print("  num_candidates: 生成数量 (notebook中建议: 1-10)")

## 🚀 4. 生成候选序列

In [ ]:
# 单个候选生成（快速测试）
if prompt_data:
    print("生成单个测试候选...")
    print(f"参数: T={generation_params['temperature']}, "
          f"结构步数={generation_params['structure_steps']}, "
          f"序列步数={generation_params['sequence_steps']}")
    
    # 创建prompt对象
    prompt = generator.create_protein(sequence=prompt_data['sequence'])
    
    # 生成
    start_time = time.time()
    
    generated = generator.chain_of_thought_generation(
        prompt,
        structure_steps=generation_params['structure_steps'],
        sequence_steps=generation_params['sequence_steps'],
        temperature=generation_params['temperature']
    )
    
    elapsed = time.time() - start_time
    
    print(f"\n✓ 生成完成！耗时: {elapsed:.1f} 秒")
    print(f"\n生成序列:")
    print(f"  长度: {len(generated.sequence)}")
    print(f"  序列: {generated.sequence[:80]}...")
    
    # 检查关键残基
    print(f"\n关键残基检查:")
    preserved = 0
    for pos, expected in KEY_RESIDUES.items():
        if pos < len(generated.sequence):
            actual = generated.sequence[pos]
            match = "✓" if actual == expected else "✗"
            print(f"  位置 {pos+1}: 期望={expected}, 实际={actual} {match}")
            if actual == expected:
                preserved += 1
    
    preservation_rate = preserved / len(KEY_RESIDUES)
    print(f"\n关键残基保留率: {preservation_rate:.1%}")
    
    # 序列复杂度
    aa_counts = Counter(generated.sequence)
    print(f"\n氨基酸分布 (Top 5):")
    for aa, count in aa_counts.most_common(5):
        print(f"  {aa}: {count} ({count/len(generated.sequence)*100:.1f}%)")
    
    # 保存
    test_candidate = generated
else:
    print("请先加载prompt数据")

In [ ]:
# 批量生成（可选）
def generate_multiple_candidates(num_candidates, temperature_range=None):
    """
    生成多个候选
    
    Args:
        num_candidates: 候选数量
        temperature_range: (min, max) 温度范围，如果为None则使用固定温度
    """
    if not prompt_data:
        print("请先加载prompt数据")
        return []
    
    candidates = []
    
    for i in range(num_candidates):
        # 确定温度
        if temperature_range:
            temp = np.random.uniform(temperature_range[0], temperature_range[1])
        else:
            temp = generation_params['temperature']
        
        print(f"\n[{i+1}/{num_candidates}] 生成候选 (T={temp:.2f})...")
        
        try:
            prompt = generator.create_protein(sequence=prompt_data['sequence'])
            
            generated = generator.chain_of_thought_generation(
                prompt,
                structure_steps=generation_params['structure_steps'],
                sequence_steps=generation_params['sequence_steps'],
                temperature=temp
            )
            
            candidates.append({
                'index': i,
                'protein': generated,
                'temperature': temp,
                'sequence': generated.sequence
            })
            
            print(f"  ✓ 完成 (长度={len(generated.sequence)})")
            
        except Exception as e:
            print(f"  ✗ 失败: {e}")
    
    return candidates

# 取消注释以生成多个候选（警告：耗时）
# all_candidates = generate_multiple_candidates(
#     num_candidates=generation_params['num_candidates'],
#     temperature_range=(0.5, 1.0)  # 或None使用固定温度
# )
# print(f"\n✓ 共生成 {len(all_candidates)} 个候选")

## 📊 5. 分析和可视化

In [ ]:
# 分析单个候选
if 'test_candidate' in locals():
    seq = test_candidate.sequence
    
    # 氨基酸组成
    aa_counts = Counter(seq)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 氨基酸频率分布
    aa_freq = pd.DataFrame([
        {'AA': aa, 'Count': count, 'Frequency': count/len(seq)}
        for aa, count in aa_counts.most_common()
    ])
    
    axes[0].bar(aa_freq['AA'], aa_freq['Frequency'])
    axes[0].set_xlabel('Amino Acid')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Amino Acid Composition')
    axes[0].tick_params(axis='x', rotation=45)
    
    # 序列复杂度（滑窗）
    window_size = 20
    complexities = []
    positions = []
    
    for i in range(0, len(seq) - window_size + 1, 5):
        window = seq[i:i+window_size]
        unique_aa = len(set(window))
        complexities.append(unique_aa)
        positions.append(i + window_size//2)
    
    axes[1].plot(positions, complexities, marker='o', markersize=3)
    axes[1].axhline(y=10, color='r', linestyle='--', label='Expected diversity')
    axes[1].set_xlabel('Position')
    axes[1].set_ylabel('Unique AA in window')
    axes[1].set_title(f'Sequence Complexity (window={window_size})')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 计算与模板的相似度
    if prompt_data:
        template_seq = prompt_data['template_data']['sequence']
        identity = sequence_identity(seq, template_seq)
        print(f"\n与模板序列相同性: {identity:.2%}")

In [ ]:
# 如果有多个候选，比较它们
if 'all_candidates' in locals() and len(all_candidates) > 1:
    print("比较所有候选...\n")
    
    # 创建对比表
    comparison = []
    
    for cand in all_candidates:
        seq = cand['sequence']
        aa_diversity = len(set(seq))
        
        if prompt_data:
            template_seq = prompt_data['template_data']['sequence']
            identity = sequence_identity(seq, template_seq)
        else:
            identity = None
        
        comparison.append({
            'Index': cand['index'],
            'Temperature': cand['temperature'],
            'Length': len(seq),
            'Diversity': aa_diversity,
            'Template Identity': identity
        })
    
    df_comparison = pd.DataFrame(comparison)
    print(df_comparison)
    
    # 可视化
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 温度 vs 多样性
    axes[0].scatter(df_comparison['Temperature'], df_comparison['Diversity'])
    axes[0].set_xlabel('Temperature')
    axes[0].set_ylabel('AA Diversity')
    axes[0].set_title('Temperature vs Diversity')
    axes[0].grid(True, alpha=0.3)
    
    # 温度 vs 序列相同性
    if 'Template Identity' in df_comparison.columns:
        axes[1].scatter(df_comparison['Temperature'], df_comparison['Template Identity'])
        axes[1].set_xlabel('Temperature')
        axes[1].set_ylabel('Template Identity')
        axes[1].set_title('Temperature vs Template Similarity')
        axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 💾 6. 保存结果

In [ ]:
# 保存单个候选
if 'test_candidate' in locals():
    output_dir = os.path.join(project_root, 'data', 'candidates')
    os.makedirs(output_dir, exist_ok=True)
    
    output_file = os.path.join(output_dir, 'notebook_candidate.fasta')
    
    header = f">notebook_candidate|temp_{generation_params['temperature']:.2f}|length_{len(test_candidate.sequence)}"
    save_to_fasta(test_candidate.sequence, output_file, header=header)
    
    print(f"✓ 候选已保存: {output_file}")

In [ ]:
# 保存多个候选
if 'all_candidates' in locals():
    output_dir = os.path.join(project_root, 'data', 'candidates')
    
    for cand in all_candidates:
        output_file = os.path.join(output_dir, f"notebook_candidate_{cand['index']}.fasta")
        header = f">notebook_{cand['index']}|temp_{cand['temperature']:.2f}"
        save_to_fasta(cand['sequence'], output_file, header=header)
    
    print(f"✓ {len(all_candidates)} 个候选已保存到 {output_dir}")

## 📈 7. 加载和分析已有结果

In [ ]:
# 加载评估结果（如果有）
results_file = os.path.join(project_root, 'data', 'results', 'evaluation_results.csv')

if os.path.exists(results_file):
    print("加载评估结果...")
    df_results = pd.read_csv(results_file)
    
    print(f"\n共 {len(df_results)} 个候选")
    print(f"通过: {df_results['pass'].sum()} ({df_results['pass'].sum()/len(df_results)*100:.1f}%)")
    
    print("\n基本统计:")
    print(df_results[['ptm', 'plddt', 'sequence_identity']].describe())
    
    # 可视化结果分布
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # pTM分布
    if 'ptm' in df_results.columns:
        axes[0, 0].hist(df_results['ptm'].dropna(), bins=20, edgecolor='black')
        axes[0, 0].axvline(x=0.8, color='r', linestyle='--', label='Threshold')
        axes[0, 0].set_xlabel('pTM')
        axes[0, 0].set_ylabel('Count')
        axes[0, 0].set_title('pTM Distribution')
        axes[0, 0].legend()
    
    # pLDDT分布
    if 'plddt' in df_results.columns:
        axes[0, 1].hist(df_results['plddt'].dropna(), bins=20, edgecolor='black')
        axes[0, 1].axvline(x=0.8, color='r', linestyle='--', label='Threshold')
        axes[0, 1].set_xlabel('pLDDT')
        axes[0, 1].set_ylabel('Count')
        axes[0, 1].set_title('pLDDT Distribution')
        axes[0, 1].legend()
    
    # 序列相同性分布
    if 'sequence_identity' in df_results.columns:
        axes[1, 0].hist(df_results['sequence_identity'].dropna(), bins=20, edgecolor='black')
        axes[1, 0].set_xlabel('Sequence Identity')
        axes[1, 0].set_ylabel('Count')
        axes[1, 0].set_title('Template Similarity Distribution')
    
    # pTM vs pLDDT散点图
    if 'ptm' in df_results.columns and 'plddt' in df_results.columns:
        scatter = axes[1, 1].scatter(
            df_results['ptm'], 
            df_results['plddt'],
            c=df_results['pass'].astype(int),
            cmap='RdYlGn',
            alpha=0.6
        )
        axes[1, 1].axvline(x=0.8, color='gray', linestyle='--', alpha=0.5)
        axes[1, 1].axhline(y=0.8, color='gray', linestyle='--', alpha=0.5)
        axes[1, 1].set_xlabel('pTM')
        axes[1, 1].set_ylabel('pLDDT')
        axes[1, 1].set_title('Quality Metrics')
        plt.colorbar(scatter, ax=axes[1, 1], label='Pass')
    
    plt.tight_layout()
    plt.show()
    
    # 显示Top 10
    print("\nTop 10 候选 (按pTM):")
    if 'ptm' in df_results.columns:
        top10 = df_results.nlargest(10, 'ptm')[['index', 'ptm', 'plddt', 'sequence_identity']]
        print(top10)
else:
    print("评估结果文件不存在")
    print("请先运行: python scripts/05_evaluate_candidates.py")

## 🧪 8. 高级分析

In [ ]:
# 序列比对可视化（需要多个候选）
def visualize_alignment(sequences, max_length=100):
    """
    简单的序列比对可视化
    """
    if len(sequences) < 2:
        print("需要至少2个序列")
        return
    
    # 截取前max_length个位置
    n_seqs = len(sequences)
    length = min(max_length, min(len(s) for s in sequences))
    
    # 创建对比矩阵
    alignment = np.zeros((n_seqs, length))
    
    for i, seq in enumerate(sequences):
        for j in range(length):
            # 与第一个序列比较
            if seq[j] == sequences[0][j]:
                alignment[i, j] = 1  # 相同
            else:
                alignment[i, j] = 0  # 不同
    
    # 绘制
    plt.figure(figsize=(15, max(3, n_seqs * 0.5)))
    plt.imshow(alignment, cmap='RdYlGn', aspect='auto')
    plt.xlabel('Position')
    plt.ylabel('Sequence')
    plt.title('Sequence Alignment (Green=Match, Red=Mismatch vs Seq 0)')
    plt.colorbar(label='Match')
    plt.tight_layout()
    plt.show()
    
    # 统计保守性
    conservation = alignment.mean(axis=0)
    
    plt.figure(figsize=(15, 3))
    plt.plot(conservation)
    plt.axhline(y=0.8, color='r', linestyle='--', label='80% conservation')
    plt.xlabel('Position')
    plt.ylabel('Conservation')
    plt.title('Sequence Conservation')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# 使用示例（如果有多个候选）
# if 'all_candidates' in locals() and len(all_candidates) > 1:
#     seqs = [c['sequence'] for c in all_candidates[:5]]  # 前5个
#     visualize_alignment(seqs)

## 📝 9. 导出报告

In [ ]:
# 生成简单报告
def generate_notebook_report():
    report = []
    report.append("=" * 70)
    report.append("GFP生成 - Notebook报告")
    report.append("=" * 70)
    report.append("")
    
    # 生成参数
    report.append("生成参数:")
    for key, value in generation_params.items():
        report.append(f"  {key}: {value}")
    report.append("")
    
    # 候选信息
    if 'test_candidate' in locals():
        report.append("测试候选:")
        report.append(f"  长度: {len(test_candidate.sequence)}")
        report.append(f"  序列: {test_candidate.sequence}")
        report.append("")
    
    if 'all_candidates' in locals():
        report.append(f"批量候选: {len(all_candidates)} 个")
        report.append("")
    
    report_text = "\n".join(report)
    
    # 保存
    output_file = os.path.join(project_root, 'data', 'results', 'notebook_report.txt')
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    
    with open(output_file, 'w') as f:
        f.write(report_text)
    
    print(report_text)
    print(f"\n✓ 报告已保存: {output_file}")

# 取消注释以生成报告
# generate_notebook_report()

## 💡 使用提示

1. **首次运行**：先运行前面的所有cell初始化环境
2. **调整参数**：修改 `generation_params` 字典
3. **快速测试**：使用单个候选生成
4. **批量生成**：取消注释批量生成代码
5. **分析结果**：使用可视化工具分析候选质量

**注意事项：**
- 生成过程较慢，建议先测试单个候选
- 第一次加载模型需要3-5分钟
- 大量生成建议使用命令行脚本而非notebook

**快捷键：**
- `Shift+Enter`: 运行当前cell
- `Ctrl+Enter`: 运行当前cell但不移动
- `A`: 在上方插入cell
- `B`: 在下方插入cell
- `DD`: 删除当前cell